# LeafTrack: train YOLO11-seg on Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iamoyes1/leaftrack/blob/main/notebooks/train_colab.ipynb)

Thin launcher: all logic lives in the `leaftrack` package. Select **Runtime → Change runtime type → T4 GPU**, then run all cells.
Checkpoints are written to Google Drive, so if the session disconnects, set `RESUME = True` and run again.

In [ ]:
MODEL = "yolo11n"  # or "yolo11s"
RESUME = False  # True after a disconnect: continues from last.pt on Drive
HF_REPO = ""  # e.g. "your-user/leaftrack-yolo11n-seg" to publish weights (needs HF_TOKEN secret)
DRIVE_ROOT = "/content/drive/MyDrive/leaftrack"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
%cd /content
!test -d leaftrack && git -C leaftrack pull -q || git clone -q https://github.com/iamoyes1/leaftrack.git
%cd /content/leaftrack
!pip install -q -e ".[train]"

## Data: download once (cached on Drive), convert, split

In [ ]:
!python data/scripts/download_komatsuna.py --archives {DRIVE_ROOT}/archives --out data/raw/komatsuna
!python data/scripts/convert_komatsuna.py
!python data/scripts/split_dataset.py

## Train

In [ ]:
resume_flag = "--resume" if RESUME else ""
!python train.py --config configs/train_{MODEL}.yaml --project {DRIVE_ROOT}/runs --name {MODEL} {resume_flag}

## Evaluate on the held-out test plants

In [ ]:
!python evaluate.py --weights {DRIVE_ROOT}/runs/{MODEL}/weights/best.pt --device 0 --out {DRIVE_ROOT}/eval/{MODEL}
from IPython.display import Markdown, display

display(Markdown(open(f"{DRIVE_ROOT}/eval/{MODEL}/results.md").read()))

## Optional: publish weights to the Hugging Face Hub
Add a write token as the Colab secret `HF_TOKEN` (key icon in the sidebar) and set `HF_REPO` above.

In [ ]:
if HF_REPO:
    import os

    from google.colab import userdata

    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    !python -m leaftrack.hub --weights {DRIVE_ROOT}/runs/{MODEL}/weights/best.pt --repo {HF_REPO} --metrics {DRIVE_ROOT}/eval/{MODEL}/metrics.json